In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.21.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os


base_path = "/content/drive/MyDrive/Shapes/Shapes"

print("Checking folders inside:", base_path)
if os.path.exists(base_path):
    for folder in os.listdir(base_path):
        path = os.path.join(base_path, folder)
        if os.path.isdir(path):
            # Count images
            count = len([f for f in os.listdir(path) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
            print(f"  {folder}: {count} images")
else:
    print("ERROR: Folder still not found.")

Checking folders inside: /content/drive/MyDrive/Shapes/Shapes
  Square: 584 images
  Overlapped pencils: 445 images
  Wave: 0 images
  Triangle: 0 images
  Circle: 31 images
  Overlapped circle: 0 images
  Star: 477 images
  Diagonal: 0 images


In [ ]:
import os


zip_path = "/content/drive/MyDrive/Shapes/Shapes.zip"

print("Unzipping the dataset to Colab's fast memory. This may take 1-2 minutes...")
!unzip -q "{zip_path}" -d /content/

# Verify the folders
print("\nChecking folders:")
base_path = "/content/Shapes/Shapes"
if not os.path.exists(base_path):
    base_path = "/content/Shapes"

for folder in os.listdir(base_path):
    path = os.path.join(base_path, folder)
    if os.path.isdir(path):
        count = len([f for f in os.listdir(path) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
        print(f"  {folder}: {count} images")

Unzipping the dataset to Colab's fast memory. This may take 1-2 minutes...

Checking folders:
  Wave: 550 images
  Circle: 575 images
  Square: 584 images
  Overlapped pencils: 445 images
  Diagonal: 554 images
  Overlapped circle: 534 images
  Star: 477 images
  Triangle: 580 images


In [ ]:
import os

print("Looking for the 'Circle' folder inside /content/ ...")
for root, dirs, files in os.walk("/content"):
    if "Circle" in dirs:
        path = os.path.join(root, "Circle")
        # Check if there are actual images inside
        count = len([f for f in os.listdir(path) if f.endswith('.png')])
        if count > 0:
            print(f"FOUND THE DATASET AT: {root}")
            print(f"Circle has {count} images here.")
            break

Looking for the 'Circle' folder inside /content/ ...
FOUND THE DATASET AT: /content/Shapes
Circle has 575 images here.


In [ ]:
import os

bad_images = [
    # Wave (Triangles)
    "img1141-W-4.png", "img1182-W-4.png", "img4051-W-4.png", "img4293-W-4.png",
    "img1955-W-3(1011).png", "img3930-W-3(1011).png",
    # Triangle (Diamonds)
    "img2489-T-5.png", "img3000-T-5.png", "img4294-T-5.png", "img2251-T-5.png",
    "img2302-T-0.png", "img1183-T-5.png", "img1706-T-5.png",
    # Star (Scribbles)
    "img3164-ST-3(11001).png", "img3054-ST-4(11101).png",
    # Diagonal (Stars/Scribbles)
    "img1421-D-5.png", "img1549-D-5.png", "img1957-D-5.png",
    "img2303-D-0.png", "img2305-D-4(11011).png", "img4067-D-5.png",
    "img3053-D-5.png", "img2185-D-0.png",
    # Overlapped Circles (Waves)
    "img1954-O-5(101111).png", "img4292-O-6.png"
]

print(f"Deleting {len(bad_images)} bad images...")
deleted_count = 0

for root, dirs, files in os.walk("/content/Shapes"):
    for file in files:
        if file in bad_images:
            os.remove(os.path.join(root, file))
            deleted_count += 1

print(f"Deleted: {deleted_count} images.")

Deleting 25 bad images...
Deleted: 25 images.


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

base_dir = "/content/Shapes"
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False
)

train_gen = datagen.flow_from_directory(
    base_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training'
)

val_gen = datagen.flow_from_directory(
    base_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation'
)

Found 3422 images belonging to 8 classes.
Found 852 images belonging to 8 classes.


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os


print("Building ResNet-50 for Fine-Tuning...")
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(128, 128, 3))


base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(train_gen.num_classes, activation='softmax')
])

# 2. Compile with a very low learning rate (0.00001) to learn carefully
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.00001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 3. Set up saving to Google Drive
save_path = "/content/drive/MyDrive/Shapes/resnet50_finetuned.keras"
checkpoint = ModelCheckpoint(save_path, monitor='val_accuracy', save_best_only=True, verbose=1)

# 4. Train for 15 Epochs
print("\nStarting Fine-Tuning Training (15 Epochs)...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True), checkpoint]
)

print("\nFine-Tuning Complete. Best model saved to Drive.")

Building ResNet-50 for Fine-Tuning...

Starting Fine-Tuning Training (15 Epochs)...
Epoch 1/15
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 609ms/step - accuracy: 0.2737 - loss: 2.0277
Epoch 1: val_accuracy improved from None to 0.14789, saving model to /content/drive/MyDrive/Shapes/resnet50_finetuned.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Shapes/resnet50_finetuned.keras
107/107 ━━━━━━━━━━━━━━━━━━━━ 133s 939ms/step - accuracy: 0.3796 - loss: 1.7513 - val_accuracy: 0.1479 - val_loss: 2.0937
Epoch 2/15
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 544ms/step - accuracy: 0.6264 - loss: 1.1966
Epoch 2: val_accuracy improved from 0.14789 to 0.50000, saving model to /content/drive/MyDrive/Shapes/resnet50_finetuned.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Shapes/resnet50_finetuned.keras
107/107 ━━━━━━━━━━━━━━━━━━━━ 75s 698ms/step - accuracy: 0.6689 - loss: 1.0762 - val_accuracy: 0.5000 - val_loss: 1.5404
Epoch 3/15
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 555ms/step - accuracy: 0.

In [ ]:
import json
import os

# Save the results to Google Drive
results = {
    "model": "ResNet-50 (Fine-Tuned)",
    "best_val_accuracy": 0.86502,
    "best_train_accuracy": max(history.history['accuracy']),
    "total_epochs_run": len(history.history['accuracy']),
    "total_images_train": train_gen.samples,
    "total_images_val": val_gen.samples
}

save_path = "/content/drive/MyDrive/Shapes/resnet50_results.json"
with open(save_path, 'w') as f:
    json.dump(results, f, indent=4)

print(f"Results saved successfully to: {save_path}")
print(f"Best Val Accuracy: {results['best_val_accuracy'] * 100:.2f}%")

Results saved successfully to: /content/drive/MyDrive/Shapes/resnet50_results.json
Best Val Accuracy: 86.50%


In [ ]:
import cv2
import numpy as np
import os
import math
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, accuracy_score
import joblib

# 1. Setup paths
base_dir = "/content/Shapes"
classes = ['Circle', 'Diagonal', 'Overlapped circle', 'Overlapped pencils', 'Square', 'Star', 'Triangle', 'Wave']

print("Extracting features from clean images... This will take ~2 minutes.")
X = []
y = []
shape_names = []

def extract_features(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    img = cv2.resize(img, (128, 128), interpolation=cv2.INTER_AREA)
    img = img[12:-12, 12:-12]

    blur = cv2.GaussianBlur(img, (5, 5), 0)
    _, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = np.ones((3, 3), np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    ink_mask = binary > 0
    ink_ratio = float(ink_mask.mean())
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if len(contours) == 0:
        return [0.0]*21 # Return 21 zeros if empty

    largest = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(largest)
    perimeter = cv2.arcLength(largest, True)
    circularity = (4 * math.pi * area / (perimeter ** 2)) if perimeter > 0 else 0.0
    hull = cv2.convexHull(largest)
    hull_area = cv2.contourArea(hull)
    solidity = area / hull_area if hull_area > 0 else 0.0
    x, y, w, h = cv2.boundingRect(largest)
    bbox = w * h
    extent = area / bbox if bbox > 0 else 0.0
    aspect_ratio = w / h if h > 0 else 0.0
    hu = cv2.HuMoments(cv2.moments(largest)).flatten()
    hu_log = [float(-math.copysign(1.0, v) * math.log10(abs(v) + 1e-30)) for v in hu]

    # We will use the first 17 features for simplicity in this pipeline
    return [ink_ratio, area, perimeter, circularity, solidity, extent, aspect_ratio] + hu_log + [len(contours), 0.0, 0.0, 0.0, 0.0]

# 2. Loop through all folders and extract
for folder in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder)
    if not os.path.isdir(folder_path): continue

    for file in os.listdir(folder_path):
        if not file.endswith('.png'): continue

        # Extract score from filename (e.g., img1003-C-3(1011).png -> 3)
        try:
            score = int(file.split('-')[-1].split('(')[0].split('.')[0])
        except:
            continue

        feats = extract_features(os.path.join(folder_path, file))
        if feats:
            X.append(feats)
            y.append(score)
            shape_names.append(folder)

X = np.array(X)
y = np.array(y)
shape_names = np.array(shape_names)
print(f"Successfully extracted features for {len(X)} images.")

Extracting features from clean images... This will take ~2 minutes.
Successfully extracted features for 4273 images.


In [ ]:
import joblib
from sklearn.ensemble import RandomForestRegressor
import os

print("\nTraining Random Forest models per shape...")

# We will train 8 separate Random Forest models, one for each shape
models_dict = {}
for shape in classes:
    # Get data for this shape only
    mask = (shape_names == shape)
    X_shape = X[mask]
    y_shape = y[mask]

    if len(X_shape) == 0: continue

    print(f"Training on {shape}: {len(X_shape)} images...")

    # Train a Random Forest
    rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
    rf.fit(X_shape, y_shape)

    # Save the model to Google Drive
    save_path = f"/content/drive/MyDrive/Shapes/rf_{shape.replace(' ', '_')}.joblib"
    joblib.dump(rf, save_path)
    models_dict[shape] = rf

print(f"\nAll {len(models_dict)} Random Forest models trained and saved to Drive!")


Training Random Forest models per shape...
Training on Circle: 575 images...
Training on Diagonal: 546 images...
Training on Overlapped circle: 532 images...
Training on Overlapped pencils: 445 images...
Training on Square: 584 images...
Training on Star: 474 images...
Training on Triangle: 573 images...
Training on Wave: 544 images...

All 8 Random Forest models trained and saved to Drive!


In [ ]:
import numpy as np
import cv2
import math
import joblib
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image as keras_image
import os

print("Loading ResNet-50 Classifier...")
# Load the fine-tuned model we saved earlier
classifier = load_model("/content/drive/MyDrive/Shapes/resnet50_finetuned.keras")

# The classes must be in the exact order that flow_from_directory used
# flow_from_directory sorts folders alphabetically:
class_names = ['Circle', 'Diagonal', 'Overlapped circle', 'Overlapped pencils',
               'Square', 'Star', 'Triangle', 'Wave']

# Load all 8 Random Forest models
print("Loading Random Forest scoring models...")
scoring_models = {}
for shape in class_names:
    model_path = f"/content/drive/MyDrive/Shapes/rf_{shape.replace(' ', '_')}.joblib"
    scoring_models[shape] = joblib.load(model_path)

print("All models loaded successfully!\n")

def extract_features_for_pipeline(image_path):
    """Extract the exact same 21 features we used to train the Random Forest."""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    img = cv2.resize(img, (128, 128), interpolation=cv2.INTER_AREA)
    img = img[12:-12, 12:-12]

    blur = cv2.GaussianBlur(img, (5, 5), 0)
    _, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = np.ones((3, 3), np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    ink_mask = binary > 0
    ink_ratio = float(ink_mask.mean())
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if len(contours) == 0:
        return [0.0]*21

    largest = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(largest)
    perimeter = cv2.arcLength(largest, True)
    circularity = (4 * math.pi * area / (perimeter ** 2)) if perimeter > 0 else 0.0
    hull = cv2.convexHull(largest)
    hull_area = cv2.contourArea(hull)
    solidity = area / hull_area if hull_area > 0 else 0.0
    x, y, w, h = cv2.boundingRect(largest)
    bbox = w * h
    extent = area / bbox if bbox > 0 else 0.0
    aspect_ratio = w / h if h > 0 else 0.0
    hu = cv2.HuMoments(cv2.moments(largest)).flatten()
    hu_log = [float(-math.copysign(1.0, v) * math.log10(abs(v) + 1e-30)) for v in hu]

    return [ink_ratio, area, perimeter, circularity, solidity, extent, aspect_ratio] + hu_log + [len(contours), 0.0, 0.0, 0.0, 0.0]

def full_pipeline(image_path):
    """Takes an image, predicts shape with ResNet, then score with Random Forest."""
    print(f"\nAnalyzing: {os.path.basename(image_path)}")
    print("-" * 40)

    # --- STEP 1: Classify the shape ---
    img = keras_image.load_img(image_path, target_size=(128, 128))
    img_array = keras_image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    shape_preds = classifier.predict(img_array, verbose=0)
    predicted_shape_idx = np.argmax(shape_preds)
    predicted_shape = class_names[predicted_shape_idx]
    shape_confidence = shape_preds[0][predicted_shape_idx] * 100
    print(f"Phase 1 (Shape Classification): {predicted_shape} (Confidence: {shape_confidence:.2f}%)")

    # --- STEP 2: Score the shape ---
    features = extract_features_for_pipeline(image_path)
    if features is None:
        print("Error: Could not extract features.")
        return

    rf_model = scoring_models[predicted_shape]
    predicted_score_raw = rf_model.predict([features])[0]
    predicted_score = int(round(predicted_score_raw))
    # Ensure score is not negative
    predicted_score = max(0, predicted_score)

    print(f"Phase 2 (Score Prediction): {predicted_score} (Raw: {predicted_score_raw:.2f})")
    print("-" * 40)
    print(f"FINAL RESULT: {predicted_shape} with score {predicted_score}")


Loading ResNet-50 Classifier...
Loading Random Forest scoring models...
All models loaded successfully!



In [ ]:
import random
import os

# Test on 4 random images from different shapes
test_folders = ["Circle", "Diagonal", "Star", "Wave"]

for folder in test_folders:
    folder_path = f"/content/Shapes/{folder}"
    if not os.path.exists(folder_path):
        continue

    # Pick a random image from this folder
    all_images = [f for f in os.listdir(folder_path) if f.endswith('.png')]
    if not all_images: continue

    random_image = random.choice(all_images)
    test_image_path = os.path.join(folder_path, random_image)

    print(f"\n{'='*50}")
    print(f"Folder: {folder}")
    full_pipeline(test_image_path)


Folder: Circle

Analyzing: img3201-C-4.png
----------------------------------------
Phase 1 (Shape Classification): Circle (Confidence: 99.03%)
Phase 2 (Score Prediction): 4 (Raw: 3.97)
----------------------------------------
FINAL RESULT: Circle with score 4

Folder: Diagonal

Analyzing: img20-D-5.png
----------------------------------------
Phase 1 (Shape Classification): Diagonal (Confidence: 99.49%)
Phase 2 (Score Prediction): 5 (Raw: 4.99)
----------------------------------------
FINAL RESULT: Diagonal with score 5

Folder: Star

Analyzing: img1850-ST-4(11011).png
----------------------------------------
Phase 1 (Shape Classification): Star (Confidence: 99.76%)
Phase 2 (Score Prediction): 4 (Raw: 4.08)
----------------------------------------
FINAL RESULT: Star with score 4

Folder: Wave

Analyzing: img2332-W-4.png
----------------------------------------
Phase 1 (Shape Classification): Wave (Confidence: 99.93%)
Phase 2 (Score Prediction): 4 (Raw: 3.70)
-------------------------